# forensic-synth — lanceur Colab
Duplique ce notebook par job/compte. Change seulement **CONFIG** et **STAGE** ci-dessous.
Tout est sauvé sur Drive et **reprenable** après coupure.

Ordre du pipeline et dépendances entre étages :
1. `partition` — aucune dépendance (charge data/blocks.json, valide contre scface_root).
2. `train_generator` — dépend de `partition` (paires réelles du Bloc A).
3. `generate` — dépend du checkpoint de `train_generator` (génère depuis les mugshots du Bloc B).
4. `fidelity` — dépend de `generate` (compare synthétique vs réel, Bloc B). Garde-fou go/no-go.
5. `train_recognition` — dépend de `generate` si condition `synthetic`/`mixed` (sinon juste `partition` pour la condition `real`).
6. `evaluate` — dépend des checkpoints de `train_recognition` (rank-1 sur Bloc C, vierge).


In [ ]:
# 1) Monter Google Drive (mémoire persistante)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) Parametres du job  ---  A MODIFIER par compte/session
CONFIG = 'configs/visible_d1.yaml'   # plus tard : configs/ir_d1.yaml, etc.
STAGE  = 'smoke'                     # smoke|partition|train_generator|generate|fidelity|train_recognition|evaluate
REPO_URL = 'https://github.com/MarouaneAyech/foresynth_facerecog.git'
CODE_DIR = '/content/foresynth_facerecog'        # checkout du code (stockage local VM, rapide, ephemere)
DATA_ROOT = '/content/drive/MyDrive/forensic-synth'  # racine des DONNEES/artefacts (Drive, persistante)


In [ ]:
# 3) Recuperer/mettre a jour le CODE (toujours dans CODE_DIR, separe des donnees)
# Code et donnees sont VOLONTAIREMENT separes : cloner/pull du code directement sur Drive
# est lent et fragile (beaucoup de petits fichiers), et un dossier Drive existant (donnees)
# n'est pas forcement un repo git -> 'git pull' y echouerait. CODE_DIR est recree a chaque
# session (stockage local de la VM, perdu a la coupure, sans consequence : c'est juste du code).
import os, shutil, subprocess

# Se placer hors de CODE_DIR AVANT toute suppression : si une execution precedente de cette
# cellule a deja fait os.chdir(CODE_DIR), rmtree(CODE_DIR) supprimerait le cwd du processus
# lui-meme (-> 'fatal: Unable to read current working directory' sur tout subprocess ensuite).
os.chdir('/content')

if os.path.exists(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', '-C', CODE_DIR, 'pull'], check=True)
else:
    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)
    subprocess.run(['git', 'clone', REPO_URL, CODE_DIR], check=True)
os.chdir(CODE_DIR)

# Racine unique des artefacts/donnees (cf. src/config.py) : seul levier a toucher pour
# changer d'environnement (cf. CLAUDE.md). Reste sur Drive, independant de CODE_DIR.
os.environ['FORENSIC_SYNTH_ROOT'] = DATA_ROOT
print('Dossier code :', os.getcwd())
print('Racine donnees (FORENSIC_SYNTH_ROOT) :', DATA_ROOT)


In [ ]:
# 4) Dépendances (figer les versions une fois stables)
!pip -q install -r requirements.txt
# Pas sur PyPI (fournit CLIPTextModelWrapper, project_face_embs utilisés par
# src/generator/base_arc2face.py) :
!pip -q install git+https://github.com/foivospar/Arc2Face.git


In [ ]:
# 5) Vérifier le GPU (les étages lourds en ont besoin ; smoke/partition non)
import torch
print('CUDA dispo :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


### Assets externes a deposer manuellement
- **antelopev2** : lien de telechargement direct + instructions dans le README
  officiel d'Arc2Face (github.com/foivospar/Arc2Face) -- decompresser sous
  `~/.insightface/models/antelopev2`, supprimer le fichier `glintr100.onnx`
  (backbone par defaut d'insightface, pas celui attendu), garder `arcface.onnx`.
- **arcface_ms1mv3_r50.pth** : checkpoint IResNet-50 entraine ArcFace/MS1MV3
  (model zoo officiel InsightFace, projet `arcface_torch`) -- a deposer sous le
  chemin `paths.arcface_weights` (cf. `configs/base.yaml`).


In [ ]:
# 6) Vérifier les assets externes À DÉPOSER MANUELLEMENT (pas de téléchargement
#    automatique ici : voir les liens officiels dans CLAUDE.md / les READMEs upstream).
from pathlib import Path
from src.config import load_config

_cfg = load_config(CONFIG)
_checks = {
    'Poids ArcFace iresnet50/arcface_ms1mv3 (generator/identity_loss, fidelity, recognition)':
        Path(_cfg['paths']['arcface_weights']),
    "Pack insightface 'antelopev2' (extraction de l'embedding ID, cf. README Arc2Face)":
        Path.home() / '.insightface' / 'models' / 'antelopev2',
}
for label, path in _checks.items():
    status = 'OK' if path.exists() else 'MANQUANT'
    print(f'[{status}] {label} -> {path}')


In [ ]:
# 7) (Optionnel) câblage d'abord — gratuit, sans GPU
!python -m pytest -q


In [ ]:
# 8) Lancer l'étage demandé (reprend automatiquement sur checkpoint)
!python -m experiments.run --config $CONFIG --stage $STAGE


## Répartition multi-comptes (jobs INDÉPENDANTS)
- Compte 1 : `STAGE='partition'` puis `STAGE='train_generator'` (lourd, 1 fois) → dataset caché sur Drive.
- Compte 1 (suite) : `STAGE='generate'` puis `STAGE='fidelity'` (go/no-go avant de continuer).
- Comptes 2/3/4 : `STAGE='train_recognition'` pour des seeds/conditions différents (jobs séparés, modifie `cfg['recognition']['conditions']`/`cfg['seeds']` ou lance plusieurs notebooks).
- Puis `STAGE='evaluate'` (agrège automatiquement les seeds disponibles par condition).

_Aucune fusion automatique entre comptes : tu ouvres chaque session manuellement._
